# Bronze Ingestion

### Importing Pyspark library

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### Checking column names inside orders

In [0]:
file_path = "/Volumes/ecommerce_lakehouse/raw/oltp_landing/orders/olist_orders_dataset.csv"

column_names = spark.read.option("header", "true").csv(file_path).limit(0).columns
print(column_names)

### Getting few rows to check data types before making schema

In [0]:
data_check = spark.read \
                .format("csv") \
                .option("header", "true") \
                .option("inferschema","true") \
                .load(file_path) \
                .limit(5)


display(data_check)

In [0]:
column_names = (spark.read
                .format("csv")             
                .option("header", "true") 
                .load(file_path)           
                .limit(0).columns)
print(column_names)

### Creating Orders schema

In [0]:

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

In [0]:
display(orders_schema)

### Adding Autoloader
read => It checks folder once and read whatever file there and stops.If we add another file later it won't read it unless we run notebook again. \
readstream => This continously watches folder. As soon as new file comes this automatically process that. \
cloudFiles => this works as a auto loader. Suppose we have 1,000 files and we add 1 more, Auto Loader is smart enough to only read the 1 new file. \
It also keeps a log of what files it has already finished. Even if the cluster crashes, it remembers exactly where it left off.

In [0]:
orders_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(orders_schema)
        .load(
            "/Volumes/ecommerce_lakehouse/raw/oltp_landing/orders/"
        )
)

### Added Metadata columns

In [0]:
bronze_orders_df = orders_stream_df \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "load_date",
        current_date()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

### Creating checkpoints

In [0]:
query = (
    bronze_orders_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/ecommerce_lakehouse/raw/checkpoints/orders/"
        )
        .trigger(availableNow=True)
        .toTable(
            "ecommerce_lakehouse.bronze.orders_raw"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.bronze.orders_raw
LIMIT 10;

In [0]:
%sql
SELECT source_file,
       COUNT(*)
FROM ecommerce_lakehouse.bronze.orders_raw
GROUP BY source_file;